# Project 2 — CORES for Monocular Depth Estimation

Kaggle-first implementation for studying convolutional-response OOD detection in a monocular depth estimation network.

**Planned domains:** NYU Depth v2 (ID) and KITTI (OOD).  
**Planned model:** FastDepth.  
**Main metrics:** AUROC, FPR95, RMSE, AbsRel, δ1, δ2, δ3.

> Start with `QUICK_MODE = True`. Dataset-specific code will be enabled after the exact Kaggle dataset sources and layouts have been selected.

## 1. Imports

In [ ]:
from __future__ import annotations

import json
import os
import platform
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')

## 2. Environment and configuration

In [ ]:
def detect_environment() -> str:
    if Path('/kaggle').exists():
        return 'kaggle'
    if 'google.colab' in sys.modules:
        return 'colab'
    return 'local'


ENVIRONMENT = detect_environment()

if ENVIRONMENT == 'kaggle':
    INPUT_ROOT = Path('/kaggle/input')
    WORK_ROOT = Path('/kaggle/working/cores-mde')
elif ENVIRONMENT == 'colab':
    INPUT_ROOT = Path('/content/data')
    WORK_ROOT = Path('/content/cores-mde')
else:
    INPUT_ROOT = Path.cwd().parent / 'data'
    WORK_ROOT = Path.cwd().parent / 'outputs'

WORK_ROOT.mkdir(parents=True, exist_ok=True)
(WORK_ROOT / 'checkpoints').mkdir(exist_ok=True)
(WORK_ROOT / 'figures').mkdir(exist_ok=True)
(WORK_ROOT / 'results').mkdir(exist_ok=True)

print(f'Environment: {ENVIRONMENT}')
print(f'Input root: {INPUT_ROOT}')
print(f'Work root: {WORK_ROOT}')

In [ ]:
@dataclass(frozen=True)
class Config:
    seed: int = 42
    quick_mode: bool = True
    train_model: bool = False
    load_checkpoint: bool = True
    image_height: int = 224
    image_width: int = 304
    batch_size: int = 8
    num_workers: int = 2
    epochs: int = 2
    learning_rate: float = 1e-4
    nyu_dataset_dir: str = 'CHANGE_ME_NYU_DATASET'
    kitti_dataset_dir: str = 'CHANGE_ME_KITTI_DATASET'


CFG = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(json.dumps(asdict(CFG), indent=2))
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')
else:
    print('WARNING: GPU not detected. Enable a GPU accelerator for training.')

## 3. Reproducibility

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # Determinism improves reproducibility but may reduce performance.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(CFG.seed)
print(f'Global seed set to {CFG.seed}.')

## 4. Runtime diagnostics

In [ ]:
def runtime_report() -> dict[str, Any]:
    report: dict[str, Any] = {
        'environment': ENVIRONMENT,
        'platform': platform.platform(),
        'python': sys.version.split()[0],
        'torch': torch.__version__,
        'numpy': np.__version__,
        'sklearn': sklearn.__version__,
        'device': str(DEVICE),
        'cuda_available': torch.cuda.is_available(),
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        report.update({
            'gpu': props.name,
            'gpu_memory_gib': round(props.total_memory / 1024**3, 2),
            'cuda': torch.version.cuda,
        })
    return report


RUNTIME = runtime_report()
print(json.dumps(RUNTIME, indent=2))
with (WORK_ROOT / 'runtime.json').open('w', encoding='utf-8') as file:
    json.dump(RUNTIME, file, indent=2)

## 5. Dataset discovery

This section deliberately fails early when dataset names have not been configured. It prevents a long Kaggle run from silently using the wrong folders.

In [ ]:
def list_input_datasets(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(path for path in root.iterdir() if path.is_dir())


available_datasets = list_input_datasets(INPUT_ROOT)
print('Attached input datasets:')
for path in available_datasets:
    print(f'  - {path.name}')
if not available_datasets:
    print('  (none found — expected before Kaggle datasets are attached)')

In [ ]:
NYU_ROOT = INPUT_ROOT / CFG.nyu_dataset_dir
KITTI_ROOT = INPUT_ROOT / CFG.kitti_dataset_dir

dataset_status = pd.DataFrame([
    {'dataset': 'NYU Depth v2', 'path': str(NYU_ROOT), 'found': NYU_ROOT.exists()},
    {'dataset': 'KITTI', 'path': str(KITTI_ROOT), 'found': KITTI_ROOT.exists()},
])
display(dataset_status)

DATASETS_READY = bool(dataset_status['found'].all())
if not DATASETS_READY:
    print('Dataset loaders remain disabled until the Config folder names are updated.')

## 6. Core validation utilities

In [ ]:
def validate_rgb_depth_pair(image: torch.Tensor, depth: torch.Tensor) -> None:
    if image.ndim != 3 or image.shape[0] != 3:
        raise ValueError(f'Expected RGB tensor [3, H, W], got {tuple(image.shape)}')
    if depth.ndim not in (2, 3):
        raise ValueError(f'Expected depth tensor [H, W] or [1, H, W], got {tuple(depth.shape)}')
    depth_hw = depth.shape[-2:]
    if image.shape[-2:] != depth_hw:
        raise ValueError(f'RGB/depth spatial mismatch: {image.shape[-2:]} vs {depth_hw}')
    if not torch.isfinite(image).all():
        raise ValueError('RGB tensor contains NaN or infinity.')
    if not torch.isfinite(depth).all():
        raise ValueError('Depth tensor contains NaN or infinity.')


dummy_image = torch.rand(3, CFG.image_height, CFG.image_width)
dummy_depth = torch.rand(1, CFG.image_height, CFG.image_width)
validate_rgb_depth_pair(dummy_image, dummy_depth)
print('RGB/depth validation smoke test passed.')

## 7. Next implementation milestone

After selecting the exact Kaggle dataset sources:

1. inspect their real directory layouts and metadata;
2. implement NYU and KITTI `Dataset` classes;
3. visualize RGB/depth pairs and valid-depth masks;
4. implement and test the depth metrics;
5. integrate FastDepth and overfit a single batch;
6. add CORES only after the depth baseline is verified.